# Fourier Transforms — Interactive Demo

This notebook walks through the theory and practice of Fourier transforms, with visualisations at every step.

**Contents**
1. What is a Fourier Transform?
2. DFT from scratch
3. FFT comparison (correctness & speed)
4. Signal decomposition
5. Frequency-domain filtering
6. 2-D FFT on images

In [ ]:
# Install dependencies only if missing
import importlib
_missing = [p for p in ["numpy", "scipy", "matplotlib", "PIL"] if not importlib.util.find_spec(p)]
if _missing:
    %pip install -q numpy scipy matplotlib Pillow

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt

# Make the local package importable
sys.path.insert(0, ".")
from fourier.dft import dft, idft, fft_analyse, generate_composite_signal, frequency_filter
from fourier.visualise import (
    plot_time_and_frequency,
    plot_signal_decomposition,
    plot_filter_comparison,
    plot_image_fft,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. What is a Fourier Transform?

Any periodic signal can be represented as a **sum of sinusoids** at different frequencies, amplitudes, and phases. The Fourier Transform decomposes a signal from the **time domain** into the **frequency domain**, revealing which frequencies are present and how strong they are.

The Discrete Fourier Transform (DFT) is defined as:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j \, 2\pi \, k \, n \, / \, N}$$

where:
- $x[n]$ is the input signal (N samples)
- $X[k]$ is the $k$-th frequency bin of the output spectrum
- $e^{-j\theta} = \cos\theta - j\sin\theta$ (Euler's formula)

## 2. DFT from Scratch

Let's build a simple signal and run our hand-written DFT on it. We'll create a signal with three frequency components: **5 Hz**, **20 Hz**, and **50 Hz**.

In [ ]:
# Create a composite signal
frequencies = [5, 20, 50]
amplitudes = [1.0, 0.5, 0.3]
sample_rate = 500.0  # Hz

t, signal, components = generate_composite_signal(
    frequencies, amplitudes, duration=1.0, sample_rate=sample_rate
)

# Run our from-scratch DFT
spectrum_manual = dft(signal)

# Plot the magnitude of our hand-written DFT
N = len(signal)
freqs_manual = np.arange(N) * sample_rate / N
magnitudes_manual = 2.0 / N * np.abs(spectrum_manual[:N // 2])

fig, ax = plt.subplots(figsize=(10, 3))
ax.stem(freqs_manual[:N // 2], magnitudes_manual, linefmt="#4C72B0", markerfmt=" ", basefmt="k-")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude")
ax.set_title("DFT from Scratch — Frequency Spectrum")
ax.set_xlim(0, sample_rate / 2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. FFT Comparison — Correctness & Speed

The Fast Fourier Transform (FFT) computes the same result as the DFT but in $O(N \log N)$ instead of $O(N^2)$. Let's verify our DFT matches NumPy's FFT, and compare their speed.

In [ ]:
# Correctness: compare our DFT to NumPy's FFT
spectrum_numpy = np.fft.fft(signal)
max_error = np.max(np.abs(spectrum_manual - spectrum_numpy))
print(f"Max absolute error between DFT and FFT: {max_error:.2e}")

# Speed: time both on increasing signal sizes
sizes = [64, 128, 256, 512, 1024]
dft_times = []
fft_times = []

for n in sizes:
    test_signal = np.random.randn(n)

    start = time.perf_counter()
    _ = dft(test_signal)
    dft_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    _ = np.fft.fft(test_signal)
    fft_times.append(time.perf_counter() - start)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, [t * 1000 for t in dft_times], "o-", label="Our DFT  O(N²)", color="#C44E52")
ax.plot(sizes, [t * 1000 for t in fft_times], "s-", label="NumPy FFT  O(N log N)", color="#55A868")
ax.set_xlabel("Signal length (samples)")
ax.set_ylabel("Time (ms)")
ax.set_title("DFT vs FFT — Execution Time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Signal Decomposition

The key insight of Fourier analysis: a complex-looking waveform is just a sum of simple sine waves. Below we visualise each individual component and the resulting composite signal.

In [ ]:
# Show each component individually, then the combined signal
fig = plot_signal_decomposition(t, signal, components, frequencies, amplitudes)
plt.show()

In [ ]:
# Now show the time-domain and frequency-domain side by side
fig = plot_time_and_frequency(t, signal, sample_rate, title="Composite Signal — Time & Frequency")
plt.show()

## 5. Frequency-Domain Filtering

One powerful application of the FFT is filtering. We can remove unwanted frequencies by zeroing out their bins in the frequency domain, then transforming back.

Below we add noise to our signal and apply a **low-pass filter** to remove the high-frequency component and noise.

In [ ]:
# Create a noisy version of our signal
t_noisy, noisy_signal, _ = generate_composite_signal(
    frequencies, amplitudes, duration=1.0, sample_rate=sample_rate, noise_std=0.4
)

# Low-pass filter: keep only frequencies below 30 Hz (removes 50 Hz component + noise)
fig = plot_filter_comparison(t_noisy, noisy_signal, sample_rate, cutoff=30.0, mode="low")
plt.show()

In [ ]:
# High-pass filter: keep only frequencies above 15 Hz (removes 5 Hz component)
fig = plot_filter_comparison(t, signal, sample_rate, cutoff=15.0, mode="high")
plt.show()

## 6. 2-D FFT on Images

Fourier transforms aren't limited to 1-D signals. A **2-D FFT** reveals spatial frequency content in images — the centre of the spectrum represents low frequencies (smooth regions), and the edges represent high frequencies (edges and texture).

We'll generate a synthetic test image with known patterns.

In [ ]:
# Create a synthetic test image: sum of 2-D sinusoidal gratings
size = 256
x = np.linspace(0, 1, size)
y = np.linspace(0, 1, size)
X, Y = np.meshgrid(x, y)

# Horizontal stripes + diagonal stripes + a circle
image = (
    np.sin(2 * np.pi * 8 * Y) +              # 8 horizontal cycles
    0.5 * np.sin(2 * np.pi * 16 * (X + Y)) +  # 16 diagonal cycles
    0.3 * np.sin(2 * np.pi * 32 * X)           # 32 vertical cycles
)

fig = plot_image_fft(image, title="Synthetic Grating — 2-D FFT")
plt.show()

In [ ]:
# You can also load a real image (greyscale) — uncomment below:
# from PIL import Image
# img = np.array(Image.open("your_image.png").convert("L"), dtype=float)
# fig = plot_image_fft(img, title="Your Image — 2-D FFT")
# plt.show()

---

## Key Takeaways

| Concept | What it shows |
|---------|--------------|
| **DFT** | Direct computation of the frequency spectrum — $O(N^2)$ but great for learning |
| **FFT** | Same result in $O(N \log N)$ — the practical algorithm used everywhere |
| **Decomposition** | Any signal = sum of sinusoids at specific frequencies and amplitudes |
| **Filtering** | Zero out frequency bins to remove unwanted components, then inverse-FFT back |
| **2-D FFT** | Extends naturally to images — reveals spatial frequency structure |

Try modifying the `frequencies`, `amplitudes`, and `cutoff` values above to explore further!